This tutorial can be followed through your browser, using Colab [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/main/tutorials/3.4_Computed_Muscle_Control.ipynb).

# 3.4 — Computed Muscle Control

Static optimization (tutorial 3.3) explains a motion frame by frame but ignores activation dynamics, so its activations do not reproduce the motion when simulated. **Computed muscle control** (CMC, [Thelen, Anderson & Delp 2003](https://doi.org/10.1016/S0021-9290(02)00432-3)) runs a forward simulation and, every control step $T$:

1. computes desired accelerations from the reference and a PD feedback on the tracking error,
$$\ddot q^* = \ddot q_\text{ref} + k_v(\dot q_\text{ref} - \dot q) + k_p(q_\text{ref} - q),\qquad k_v = 2\sqrt{k_p};$$
2. solves a static optimization for $\ddot q^*$, with each activation bounded by what activation dynamics can reach within the step;
3. inverts MuJoCo's own activation dynamics to get the excitations $u$ whose mean activation over the step is the optimised one;
4. simulates the step with $u$ (and the reserves).

Part 1 reproduces a classic OpenSim CMC example on the 2-DoF elbow; part 2 tracks the walk of tutorial 3.3 on MyoLeg.

**Prerequisites:** Notebook 3.3 · `myosuite` installed · CPU only, under a minute.

In [ ]:
import os; _nb = globals().get("__vsc_ipynb_file__"); _nb and os.chdir(os.path.dirname(_nb))  # VS Code: work relative to this notebook

#  If running in Colab, install myosuite and fetch the tutorial files
if os.getenv("COLAB_RELEASE_TAG"):
    !pip install git+https://github.com/MyoHub/myosuite
    os.environ["MUJOCO_GL"] = "egl"
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/MyoHub/myosuite myosuite
    %cd myosuite
    !git sparse-checkout set tutorials
    %cd tutorials

In [ ]:
import copy
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import myo_sim
import numpy as np
import pandas as pd

from myosuite import make_env
from myosuite.core.model_builder import ModelBuilder
from myosuite.utils.video_io import show_video, write_video

files = next(p for p in (Path("files"), Path("tutorials/files")) if p.exists())
sys.path.insert(0, str((files / "3.3").resolve()))
import opensim_gait  # noqa: E402  (the walk of tutorial 3.3)

**MyoSuite biomechanics modules.** All imports of the CMC pipeline live in this one cell.

In [ ]:
# --- MyoSuite biomechanics (kinematics, static optimization, computed muscle control) ---
from myosuite.physics.computed_muscle_control import ComputedMuscleControlConfig, computed_muscle_control
from myosuite.physics.inverse_dynamics import clip_kinematics
from myosuite.physics.static_optimization import StaticOptimizationConfig

## Part 1 — Elbow: MyoSuite vs OpenSim CMC

MyoElbow (2 DoF, 6 muscles) is MuJoCo's port of OpenSim's [arm26](https://github.com/opensim-org/opensim-models/tree/master/Models/Arm26). `files/3.4/joint_trajectory.csv` holds a 1 s reference (angles, velocities and accelerations at 500 Hz) and `files/3.4/opensim_cmc.csv` the controls OpenSim's CMC computed for it.

<img src="https://picgo-liusiyuan.oss-cn-beijing.aliyuncs.com/picgo-lsy/202504161341857.png" alt="CMC" width="1000">

In [ ]:
elbow = ModelBuilder.from_xml_file(myo_sim.get_path("legacy/elbow/myoelbow_2dof6muscles.xml")).build()[0]
ref = pd.read_csv(files / "3.4" / "joint_trajectory.csv")
q_ref = ref[["angle1", "angle2"]].to_numpy()  # shoulder, elbow [rad]
dq_ref = ref[["velocity1", "velocity2"]].to_numpy()
ddq_ref = ref[["acceleration1", "acceleration2"]].to_numpy()
dt_elbow = float(np.diff(ref["time"]).mean())  # 2 ms: one physics step per control step

t0 = time.time()
cmc_elbow = computed_muscle_control(elbow, q_ref, dq_ref, ddq_ref, dt_elbow)  # default kp = 400
print(f"CMC: {time.time() - t0:.2f} s for {len(cmc_elbow.excitation)} steps")
print(f"max tracking error: {np.degrees(np.abs(cmc_elbow.tracking_error)).max():.3f} deg")

In [ ]:
opensim = pd.read_csv(files / "3.4" / "opensim_cmc.csv", skiprows=6)
muscles = [elbow.actuator(i).name for i in range(elbow.nu)]
t_cmc = ref["time"].to_numpy()[1:]  # state at the end of each control step

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for k, (name, col) in enumerate((("shoulder", "r_shoulder_elev"), ("elbow", "r_elbow_flex"))):
    axes[k].plot(ref["time"], np.degrees(q_ref[:, k]), "k:", lw=2, label="reference")
    axes[k].plot(t_cmc, np.degrees(cmc_elbow.qpos[:, k]), label="MyoSuite CMC")
    axes[k].plot(opensim["time"], np.degrees(opensim[col]), "--", label="OpenSim CMC")
    axes[k].set(title=name, xlabel="time (s)", ylabel="angle (deg)")
axes[0].legend()
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 5), sharex=True, sharey=True)
for ax, name in zip(axes.ravel(), muscles):
    ax.plot(t_cmc, cmc_elbow.excitation[:, elbow.actuator(name).id], label="MyoSuite")
    ax.plot(opensim["time"], opensim[name], "--", label="OpenSim")
    ax.set_title(name)
axes[0, 0].legend()
for ax in axes[1]:
    ax.set_xlabel("time (s)")
fig.suptitle("muscle excitations")
fig.tight_layout()
plt.show()

Both track the reference within a fraction of a degree. The excitations differ because the muscle models (MuJoCo's vs OpenSim's Thelen muscle), the activation dynamics and the gains ($k_p = 400$ here, 100 in OpenSim) differ. Toward the end, with the elbow at 90°, the stretched long triceps pulls passively on the shoulder; the minimum-effort solution holds the shoulder with the long biceps and balances the biceps' elbow flexion with the lateral and medial triceps.

## Part 2 — Walking on estimated ground reactions

The same call tracks the MyoLeg walk of tutorial 3.3. The forward simulation has no contacts: like OpenSim, which applies measured ground reactions, CMC applies the ground reactions its static optimization estimates (`contact_forces=True`), and residual forces on the root take what the feet cannot. We track 2 s at 100 Hz (10 physics steps per control step).

In [ ]:
env = make_env("myoLegWalk-v0")
leg = copy.deepcopy(env.unwrapped.model)
env.close()
mot = opensim_gait.read_mot(files / "3.5" / "subject01_walk1_ik.mot")
qpos, _ = opensim_gait.treadmill_walk_to_qpos(leg, mot, rate=100.0)
dt = 0.01
kin = clip_kinematics(leg, qpos, dt, lowpass_hz=6.0)
clip = slice(20, 221)  # 2 s away from the filter's edge effects

config = ComputedMuscleControlConfig(
    optimization=StaticOptimizationConfig(contact_forces=True, contact_distance=0.05)
)
t0 = time.time()
cmc = computed_muscle_control(leg, kin.qpos[clip], kin.qvel[clip], kin.qacc[clip], dt, config=config)
print(f"CMC: {time.time() - t0:.1f} s for {len(cmc.excitation)} steps, "
      f"{np.mean(cmc.optimization.status == 'solved'):.0%} solved")
err = cmc.tracking_error
print(f"root: {1000 * np.abs(err[:, :3]).max():.1f} mm, {np.degrees(np.abs(err[:, 3:6])).max():.2f} deg max")
print(f"joints: {np.degrees(np.abs(err[:, 6:])).max():.2f} deg max, {np.degrees(np.sqrt(np.mean(err[:, 6:] ** 2))):.2f} deg RMS")

Excitations lead the activations they produce (activation dynamics), the estimated ground reactions carry the body, and the tracking error stays below a few degrees.

In [ ]:
time_s = np.arange(1, len(cmc.excitation) + 1) * dt
body_weight = leg.body_mass.sum() * np.linalg.norm(leg.opt.gravity)
MUSCLES = ["gasmed_r", "soleus_r", "tibant_r", "vasmed_r"]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for name, color in zip(MUSCLES, plt.cm.tab10.colors):
    i = leg.actuator(name).id
    axes[0].plot(time_s, cmc.excitation[:, i], color=color, alpha=0.4)
    axes[0].plot(time_s, cmc.activation[:, i], color=color, label=name)
axes[0].set(title="excitation (light) and activation", xlabel="time (s)", ylim=(0, 1))
axes[0].legend(fontsize="small")
for side in ("r", "l"):
    grf = sum(cmc.optimization.contact_force[:, leg.body(f"{b}_{side}").id, 2] for b in ("calcn", "toes"))
    axes[1].plot(time_s, grf / body_weight, label=f"foot {side}")
axes[1].set(title="estimated vertical ground reaction", xlabel="time (s)", ylabel="body weights")
axes[1].legend(fontsize="small")
for name in ("hip_flexion_r", "knee_angle_r", "ankle_angle_r"):
    axes[2].plot(time_s, np.degrees(err[:, leg.jnt_dofadr[leg.joint(name).id]]), label=name)
axes[2].set(title="tracking error", xlabel="time (s)", ylabel="deg")
axes[2].legend(fontsize="small")
fig.tight_layout()
plt.show()

## Video

The forward simulation driven by the CMC excitations; muscles redden with their simulated activation.

In [ ]:
frames = opensim_gait.render_walk(leg, cmc.qpos, cmc.activation, every=2)
os.makedirs("videos", exist_ok=True)
output_name = "videos/myoleg_cmc_walk.mp4"
write_video(output_name, frames, fps=50)
show_video(output_name)

## Next

- `ComputedMuscleControl(model, config).step(data, qpos_ref, qvel_ref, qacc_ref, n_substeps=...)` advances a simulation one control step at a time, e.g. on streamed kinematics.
- `max_activation` caps activations per step, e.g. `1 - MF` of a fatigue model (tutorial 4.2).
- `ComputedMuscleControlConfig(apply_reserves=False)` shows where the muscles alone fall short.